## 1.实战开发
- Agent跑通了，但目前还存在几个问题：
- 目前的图片信息还是采用base64方式提交给模型，会占用大量内存，性能差
- 我们没有开发自己的前端，用户体验不好

接下来，我们就逐一解决这些问题。

在向模型提交多模态消息，比如：音频、视频、图片时，我们不建议直接发送文件数据（base64）给模型，这会大量占用内存和会话记忆。更常见的方案是：
- 先将多模态文件上传至通用的OSS服务，例如：阿里云OSS、腾讯云COS等（OSS，就是一个网络网盘）
- 获取oss服务的文件url地址，组织多模态消息，发送给大模型

因此，我们需要单独开发一个文件上传的服务接口，让前端先上传好文件，再调用Agent.

我们的服务端需要具备以下接口：
- 对话接口：接收用户聊天消息，并调用Agent
- 会话管理接口：查询或删除会话历史
- 文件上传接口：调用OSS提供的客户端，实现文件上传授权，将来由前端完成文件上传，文件不经过服务器。

### 1.1.OSS配置
- 开通存储服务（OSS）
- 申请API_KEY，并勾选“OpenAPI 调用访问”（这代表这个账号是给代码调用的，而不是给人类登录网页用的）。
- 给钥匙授权：现在这把钥匙只是一个普通的空钥匙，什么都打不开。你需要点击“新增授权”，在权限列表里搜索 OSS，选择 AliyunOSSFullAccess（管理对象存储服务的权限）。点确定。（这一步就是告诉阿里云，拿着这把钥匙的人，可以往我的 OSS 盘里读写文件。）
- 开通 OSS 桶：有了服务和钥匙，还需要这个网盘里建了一个“根文件夹”，用来装你的项目文件。
- 配置跨域（CORS）：在新建好的 Bucket 设置里，找到“跨域设置”，点击“创建规则”。如果不配这一步，当你的前端网页尝试往阿里云上传图片时，浏览器会因为安全原因把这个行为拦截掉。
- 配置 .env 文件（把钥匙交给代码）

现在，阿里云那边的事情全部办完了。你手里有了三样东西：

OSS_ACCESS_KEY_ID（钥匙 ID）

OSS_ACCESS_KEY_SECRET（钥匙密码）

OSS_BUCKET（你刚刚建的那个桶的名字）